# Muestreo representativo en RetailRocket

## Análisis de particionamiento y extracción de submuestras con PySpark

**Base de datos D:** RetailRocket Recommender System Dataset (Kaggle).

**Objetivo de la actividad:** construir una muestra representativa de la población de usuarios de una plataforma de comercio electrónico, usando variables de comportamiento observables en `events.csv`, y dejando listo el código para extraer submuestras por partición.

**Nota metodológica:** el dataset no incluye variables demográficas ni contexto de dispositivo o ubicación, por lo que la caracterización se apoya en variables conductuales: tipo de evento, actividad del usuario, temporalidad y comportamiento de conversión.


## 1. Descripción breve de la base de datos D

El dataset incluye tres archivos: `events.csv`, `item_properties.csv` y `category_tree.csv`. El archivo principal de comportamiento contiene eventos de tipo `view`, `addtocart` y `transaction`, recopilados durante aproximadamente 4.5 meses. El documento reporta 2,756,101 eventos, 1,407,580 visitantes únicos, 2,664,312 visualizaciones, 69,332 adiciones al carrito y 22,457 transacciones. También se observa que `transactionid` sólo aparece en eventos de compra.


## 2. Variables de caracterización propuestas

| Variable | Dominio / valores típicos | Estadística conocida o derivada | Comentarios |
|---|---|---:|---|
| `event` | `view`, `addtocart`, `transaction` | `view` domina ampliamente; `transaction` es minoritaria | Es la variable más útil para separar el funnel de conversión. |
| `timestamp` | Entero Unix / fecha-hora derivada | Cobertura aproximada de 4.5 meses | Permite analizar estacionalidad, día de semana y hora. |
| `visitorid` | Identificador hash de usuario | 1,407,580 visitantes únicos | Se usa para medir intensidad de interacción y segmentar actividad. |
| `itemid` | Identificador hash de producto | 235,061 productos únicos reportados en la exploración previa | Sirve para medir popularidad y construir estratos por producto. |
| `transactionid` | Entero o nulo | Sólo existe en compras; alta proporción de nulos | Útil para identificar conversiones y no debe usarse como campo de segmentación principal. |

### Variables derivadas recomendadas
- `event_hour`, `event_dayofweek`, `event_month` a partir de `timestamp`.
- `events_per_user` a partir de `visitorid`.
- `user_activity_level` (baja / media / alta) usando cuantiles de `events_per_user`.
- `item_popularity_level` (baja / media / alta) usando conteos por `itemid`.


## 3. Estrategia de particionamiento

La partición propuesta combina **tipo de evento** con **nivel de actividad del usuario**. Esta decisión ayuda a preservar el comportamiento del funnel y, al mismo tiempo, evita que los usuarios muy activos dominen toda la muestra.

### Reglas de partición
1. **P1:** usuarios de actividad baja con eventos `view`.
2. **P2:** usuarios de actividad media con eventos `view`.
3. **P3:** usuarios de actividad alta con eventos `view`.
4. **P4:** eventos `addtocart` (independientemente del nivel de actividad, por su baja frecuencia).
5. **P5:** eventos `transaction` (censo completo o casi completo, por su rareza).

### Justificación
- `view` es la clase mayoritaria, por lo que conviene estratificarla por actividad del usuario.
- `addtocart` y `transaction` son clases raras y representan señales de intención y conversión; conviene preservarlas con alta cobertura.
- La segmentación por actividad reduce sesgo hacia usuarios hiperactivos y mejora la representatividad del comportamiento general.


In [4]:
!pip install kagglehub

In [9]:
# =========================================================
# 0) Preparación del entorno
# =========================================================
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName('RetailRocket-Muestreo-Representativo').getOrCreate()

spark.sparkContext.setLogLevel('WARN')

import kagglehub

path = kagglehub.dataset_download("retailrocket/ecommerce-dataset")


In [11]:
# =========================================================
# 1) Carga de datos
# =========================================================
# Ajusta las rutas según donde tengas los archivos descargados.

# events.csv suele venir sin encabezados en algunas versiones del dataset.
events_path = f'{path}/events.csv'
item_properties_path_1 = f'{path}/item_properties_part1.csv'
item_properties_path_2 = f'{path}/item_properties_part2.csv'
category_tree_path = f'{path}/category_tree.csv'

events = (
    spark.read.option('header', 'true')
    .option('inferSchema', 'true')
    .csv(events_path)
)

item_properties_1 = (
    spark.read.option('header', 'true')
    .option('inferSchema', 'true')
    .csv(item_properties_path_1)
)

item_properties_2 = (
    spark.read.option('header', 'true')
    .option('inferSchema', 'true')
    .csv(item_properties_path_2)
)

item_properties = item_properties_1.union(item_properties_2)

category_tree = (
    spark.read.option('header', 'true')
    .option('inferSchema', 'true')
    .csv(category_tree_path)
)

print('Events rows:', events.count())
print('Item properties rows:', item_properties.count())
print('Category tree rows:', category_tree.count())


Events rows: 2756101
Item properties rows: 20275902
Category tree rows: 1669


In [12]:
# =========================================================
# 2) Inspección inicial
# =========================================================
events.printSchema()
events.show(5, truncate=False)

print('Columnas events:', len(events.columns))
print('Registros events:', events.count())


root
 |-- timestamp: long (nullable = true)
 |-- visitorid: integer (nullable = true)
 |-- event: string (nullable = true)
 |-- itemid: integer (nullable = true)
 |-- transactionid: integer (nullable = true)

+-------------+---------+-----+------+-------------+
|timestamp    |visitorid|event|itemid|transactionid|
+-------------+---------+-----+------+-------------+
|1433221332117|257597   |view |355908|NULL         |
|1433224214164|992329   |view |248676|NULL         |
|1433221999827|111016   |view |318965|NULL         |
|1433221955914|483717   |view |253185|NULL         |
|1433221337106|951259   |view |367447|NULL         |
+-------------+---------+-----+------+-------------+
only showing top 5 rows
Columnas events: 5
Registros events: 2756101


In [13]:
# =========================================================
# 3) Valores faltantes / nulos
# =========================================================
null_counts = events.select([
    F.sum(F.col(c).isNull().cast('int')).alias(c)
    for c in events.columns
])
null_counts.show(truncate=False)


+---------+---------+-----+------+-------------+
|timestamp|visitorid|event|itemid|transactionid|
+---------+---------+-----+------+-------------+
|0        |0        |0    |0     |2733644      |
+---------+---------+-----+------+-------------+



In [14]:
# =========================================================
# 4) Estadísticos generales y distribución de eventos
# =========================================================
events.describe().show(truncate=False)

events.groupBy('event').count().orderBy(F.desc('count')).show(truncate=False)

counts = {r['event']: r['count'] for r in events.groupBy('event').count().collect()}
view_n = counts.get('view', 0)
cart_n = counts.get('addtocart', 0)
purchase_n = counts.get('transaction', 0)

total_n = events.count()
print('Proporción view:', round(view_n / total_n, 6))
print('Proporción addtocart:', round(cart_n / total_n, 6))
print('Proporción transaction:', round(purchase_n / total_n, 6))


26/05/06 17:11:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+-------+---------------------+-----------------+---------+------------------+-----------------+
|summary|timestamp            |visitorid        |event    |itemid            |transactionid    |
+-------+---------------------+-----------------+---------+------------------+-----------------+
|count  |2756101              |2756101          |2756101  |2756101           |22457            |
|mean   |1.4364244883481125E12|701922.8832292431|NULL     |234922.4783750668 |8826.497795787505|
|stddev |3.3663121800168505E9 |405687.5208087459|NULL     |134195.42521364646|5098.996289874063|
|min    |1430622004384        |0                |addtocart|3                 |0                |
|max    |1442545187788        |1407579          |view     |466867            |17671            |
+-------+---------------------+-----------------+---------+------------------+-----------------+

+-----------+-------+
|event      |count  |
+-----------+-------+
|view       |2664312|
|addtocart  |69332  |
|transaction|224

In [15]:
# =========================================================
# 5) Variables derivadas para particionamiento
# =========================================================
# Timestamp en segundos a fecha/hora legible.
# En el dataset de RetailRocket, timestamp suele estar en milisegundos.
events_enriched = (
    events
    .withColumn('event_ts', F.to_timestamp(F.from_unixtime(F.col('timestamp') / 1000)))
    .withColumn('event_date', F.to_date('event_ts'))
    .withColumn('event_hour', F.hour('event_ts'))
    .withColumn('event_dayofweek', F.date_format('event_ts', 'u').cast('int'))
)

user_activity = (
    events_enriched.groupBy('visitorid')
    .agg(F.count('*').alias('events_per_user'))
)

# Cuantiles para definir actividad baja / media / alta.
q1, q2 = user_activity.approxQuantile('events_per_user', [0.33, 0.66], 0.01)
print('Umbral baja-media:', q1)
print('Umbral media-alta:', q2)

user_activity = (
    user_activity
    .withColumn(
        'user_activity_level',
        F.when(F.col('events_per_user') <= F.lit(q1), F.lit('baja'))
         .when(F.col('events_per_user') <= F.lit(q2), F.lit('media'))
         .otherwise(F.lit('alta'))
    )
)

partitioned = events_enriched.join(
    user_activity.select('visitorid', 'events_per_user', 'user_activity_level'),
    on='visitorid',
    how='left'
)
partitioned.select('visitorid', 'event', 'events_per_user', 'user_activity_level', 'event_ts').show(5, truncate=False)


Umbral baja-media: 1.0
Umbral media-alta: 1.0


[Stage 47:=======>                                                  (1 + 7) / 8]

+---------+-----+---------------+-------------------+-------------------+
|visitorid|event|events_per_user|user_activity_level|event_ts           |
+---------+-----+---------------+-------------------+-------------------+
|475715   |view |3              |alta               |2015-09-15 10:37:26|
|514337   |view |1              |baja               |2015-07-06 15:45:23|
|800692   |view |9              |alta               |2015-05-19 08:53:10|
|892013   |view |2024           |alta               |2015-08-27 19:39:34|
|1114512  |view |2              |alta               |2015-07-23 00:51:51|
+---------+-----+---------------+-------------------+-------------------+
only showing top 5 rows


In [16]:
# =========================================================
# 6) Extracción de particiones
# =========================================================
P1 = partitioned.filter((F.col('event') == 'view') & (F.col('user_activity_level') == 'baja'))
P2 = partitioned.filter((F.col('event') == 'view') & (F.col('user_activity_level') == 'media'))
P3 = partitioned.filter((F.col('event') == 'view') & (F.col('user_activity_level') == 'alta'))
P4 = partitioned.filter(F.col('event') == 'addtocart')
P5 = partitioned.filter(F.col('event') == 'transaction')

print('P1 rows:', P1.count())
print('P2 rows:', P2.count())
print('P3 rows:', P3.count())
print('P4 rows:', P4.count())
print('P5 rows:', P5.count())


P1 rows: 998953


P2 rows: 0


P3 rows: 1665359
P4 rows: 69332
P5 rows: 22457


In [17]:
# =========================================================
# 7) Técnica de muestreo por partición
# =========================================================
# Propuesta:
# - P1, P2, P3: muestreo aleatorio simple sin reemplazo con fracción pequeña.
# - P4: fracción mayor, porque addtocart es poco frecuente.
# - P5: censo completo (100%) o muestreo muy cercano al total, porque transaction es la señal más valiosa y escasa.

sample_fractions = {
    'P1': 0.02,
    'P2': 0.03,
    'P3': 0.05,
    'P4': 0.30,
    'P5': 1.00,
}

P1_sample = P1.sample(withReplacement=False, fraction=sample_fractions['P1'], seed=42)
P2_sample = P2.sample(withReplacement=False, fraction=sample_fractions['P2'], seed=42)
P3_sample = P3.sample(withReplacement=False, fraction=sample_fractions['P3'], seed=42)
P4_sample = P4.sample(withReplacement=False, fraction=sample_fractions['P4'], seed=42)
P5_sample = P5.sample(withReplacement=False, fraction=sample_fractions['P5'], seed=42)

print('P1 sample:', P1_sample.count())
print('P2 sample:', P2_sample.count())
print('P3 sample:', P3_sample.count())
print('P4 sample:', P4_sample.count())
print('P5 sample:', P5_sample.count())


P1 sample: 20214


P2 sample: 0


P3 sample: 82927
P4 sample: 20753
P5 sample: 22457


In [18]:
# =========================================================
# 8) Unión de submuestras y verificación final
# =========================================================
final_sample = P1_sample.unionByName(P2_sample).unionByName(P3_sample).unionByName(P4_sample).unionByName(P5_sample)

print('Total final sample:', final_sample.count())
final_sample.groupBy('event').count().orderBy(F.desc('count')).show(truncate=False)

Total final sample: 146468


[Stage 130:========================>                              (11 + 8) / 25]

+-----------+------+
|event      |count |
+-----------+------+
|view       |103258|
|transaction|22457 |
|addtocart  |20753 |
+-----------+------+



## 4. Técnica de muestreo por partición

**P1, P2 y P3 (eventos `view`):** muestreo aleatorio simple sin reemplazo. Al estratificar por nivel de actividad, se evita que los usuarios con muchas visitas dominen la muestra.

**P4 (`addtocart`):** muestreo aleatorio con fracción mayor que la de `view`, porque es una clase minoritaria y representa intención de compra.

**P5 (`transaction`):** preferentemente censo completo. Si el volumen operativo obliga a reducirlo, usar una fracción muy alta.

### Argumento de representatividad
La combinación de estratificación conductual + muestreo aleatorio reduce el sesgo hacia el comportamiento mayoritario y preserva las señales raras de conversión. Esto hace que la muestra sea más útil para entrenamiento posterior de modelos de conversión y recomendación.
